# 01 — Ingest Synergy API → raw Volume

Pulls the worked-example entities (**teams**, **games**) via the OAuth client and lands each as JSON in
the `raw_data` Volume, partitioned by entity. Bronze (notebook 02) Auto-Loads from here.

- **teams** — reference, pull all
- **games** — date-scoped: set `SYNERGY_START_DATE` / `SYNERGY_END_DATE` (+ optional `SYNERGY_SEASON`) in `.env`

**TODO (SA):** fan out the rest of the entities — register them in `synergy_schemas.ENTITIES` and add a
pull below (every route is `/api/<entity>/filter`; reuse `api.filter(path, body)`).

In [ ]:
# --- dual-mode setup: runs locally via Databricks Connect AND inside a Databricks Git Folder ---
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
try:
    spark  # already present inside a Databricks workspace notebook
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "main")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "synergy_baseball_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
B, S, G = f"{UC_CATALOG}.{BRONZE_SCHEMA}", f"{UC_CATALOG}.{SILVER_SCHEMA}", f"{UC_CATALOG}.{GOLD_SCHEMA}"
print("target:", B, "|", S, "|", G)

In [ ]:
def get_secret(env_var, scope="synergy", key=None):
    """Resolve a credential from .env first, then a Databricks secret scope (so the same notebook
    runs from a laptop or inside a Databricks Git Folder). Synergy creds live under scope `synergy`,
    keys `client_id` / `client_secret` — NEVER hardcode or commit them."""
    val = os.getenv(env_var)
    if val:
        return val
    from pyspark.dbutils import DBUtils  # type: ignore
    return DBUtils(spark).secrets.get(scope=scope, key=key or env_var.lower())

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# ensure the medallion schemas + raw Volume exist (idempotent — safe to re-run)
for sch in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{sch}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}")

def upload_json(volume_path: str, payload) -> int:
    """Write a JSON payload to the Volume via the Files API (network upload, no FUSE needed)."""
    body = json.dumps(payload).encode("utf-8")
    w.files.upload(volume_path, body, overwrite=True)
    return len(body)
print("volume ready:", VOLUME_PATH)

In [ ]:
from synergy_client import SynergyAPI
api = SynergyAPI(
    client_id=get_secret("SYNERGY_CLIENT_ID", key="client_id"),
    client_secret=get_secret("SYNERGY_CLIENT_SECRET", key="client_secret"),
)

def land(entity: str, rows: list[dict], suffix: str = ""):
    """Land a list of row-dicts as one JSON file in the Volume (one folder per entity)."""
    path = f"{VOLUME_PATH}/{entity}/{entity}{('_' + suffix) if suffix else ''}.json"
    n = upload_json(path, rows)
    print(f"  landed {len(rows):>6,} {entity} rows ({n:,} bytes) -> {path}")

In [ ]:
# teams (reference)
land("teams", api.get_teams())

In [ ]:
# games (date-scoped). Defaults to a 1-week window in 2024 if not set in .env.
start = os.getenv("SYNERGY_START_DATE", "2024-04-01")
end   = os.getenv("SYNERGY_END_DATE",   "2024-04-07")
season = int(os.getenv("SYNERGY_SEASON", "2024"))
games = api.get_games(min_date=start, max_date=end, season=season)
land("games", games, suffix=f"{start}_{end}")

Re-run with a different window to backfill more games — each window lands its own file, and the bronze Auto Loader picks up only new files.